Historical plotting notebook. Start Jupyter from the repository root; use the experiment READMEs and command-line runners to generate inputs first. Cells retain exploratory plotting variants; Run All is not the reproduction entry point.

In [ ]:
from pathlib import Path
import os, sys
while not Path("pyproject.toml").exists():
    if Path.cwd() == Path.cwd().parent: raise RuntimeError("Start Jupyter inside this repository")
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))


# Evolution of GIs During Training (MNIST)

- Panel A: Test accuracy vs. training epochs
- Panel B: Mean positive GI growth between training samples over epochs
- Panel C: Average GI along RSV directions over epochs
- Panel D: Average GI along random directions over epochs

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import re
import os
from tqdm import tqdm
import matplotlib.image as mpimg
import glob

In [ ]:
# Create output directory
directory = "exp1_plots"
os.makedirs(directory, exist_ok=True)

In [ ]:
# Set global plotting style parameters for consistency
plt.rcParams['figure.figsize'] = (6, 3)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Liberation Sans']
plt.rcParams['font.size'] = 16
plt.rcParams['axes.titlesize'] = 20
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['xtick.labelsize'] = 16
plt.rcParams['ytick.labelsize'] = 16
plt.rcParams['legend.fontsize'] = 16
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False



In [ ]:
# Helper function to create standardized epoch labels
def create_epoch_labels(num_epochs):
    """Create standardized epoch labels: E0, E1, E2, etc."""
    return ['E0'] + [f'E{i}' for i in range(1, num_epochs)]

In [ ]:
def plot_training_test_accuracy(train_csv_path, test_csv_path):
    # Read the CSV files
    train_df = pd.read_csv(train_csv_path)
    test_df = pd.read_csv(test_csv_path)
    
    # Filter only the end of epoch checkpoints (batch 399 for epochs 0-9) and epoch0_batch0
    checkpoints_to_keep = ['epoch0_batch0'] + [f'epoch{i}_batch399' for i in range(10)]
    train_df = train_df[train_df['checkpoint'].isin(checkpoints_to_keep)]
    test_df = test_df[test_df['checkpoint'].isin(checkpoints_to_keep)]
    
    # Create a simple epoch number for x-axis
    train_df['epoch'] = np.arange(len(train_df))
    test_df['epoch'] = np.arange(len(test_df))
    
    # Create a figure with clean style
    plt.figure(figsize=(10, 6), facecolor='white')
    
    # Plot training accuracy
    plt.plot(train_df['epoch'], train_df['total_accuracy'], 'o-', 
             color='#1f77b4', linewidth=2.5, markersize=8, label='Training Accuracy')
    
    # Plot test accuracy
    plt.plot(test_df['epoch'], test_df['total_accuracy'], 's-', 
             color='#ff7f0e', linewidth=2.5, markersize=8, label='Test Accuracy')
    
    # Create standardized x-axis labels
    x_labels = create_epoch_labels(len(train_df))
    
    # Customize the plot
    plt.title('Training and Test Accuracy Over Epochs')#, fontsize=16)
    plt.xlabel('Epoch') #, fontsize=14)
    plt.ylabel('Accuracy (%)') #, fontsize=14)
    plt.xticks(np.arange(len(x_labels)), x_labels)
    plt.grid(axis='y', linestyle='--', alpha=0.3)
    plt.legend()
    
    plt.savefig(os.path.join(directory, 'panel_a_accuracy.pdf'), 
               bbox_inches='tight', facecolor='white')
    
    # Save the figure
    plt.savefig(os.path.join(directory, 'panel_a_accuracy.png'), dpi=300, bbox_inches='tight', facecolor='white')
    
    # Show the plot
    plt.show()

# Usage
train_csv_path = "checkpoints/mnist_original_mlp/mnist_train_checkpoints.csv"
test_csv_path = "checkpoints/mnist_original_mlp/mnist_test_checkpoints.csv"
plot_training_test_accuracy(train_csv_path, test_csv_path)

In [ ]:
def plot_train_train_gi(analysis_output_dir):
    """Plot the evolution of GI between training samples."""
    
    # Find all checkpoint directories
    checkpoint_dirs = sorted(glob.glob(os.path.join(analysis_output_dir, "model_epoch*")))
    if not checkpoint_dirs:
        checkpoint_dirs = [os.path.join(analysis_output_dir, d) for d in os.listdir(analysis_output_dir) 
                          if os.path.isdir(os.path.join(analysis_output_dir, d)) and d.startswith('model_epoch')]

    print(f"Found {len(checkpoint_dirs)} checkpoint directories")

    # Store checkpoint info
    checkpoints = []
    for cp_dir in checkpoint_dirs:
        cp_name = os.path.basename(cp_dir)
        epoch_match = re.search(r'model_epoch(\d+)', cp_name)
        batch_match = re.search(r'batch(\d+)', cp_name)
        
        epoch = int(epoch_match.group(1)) if epoch_match else 0
        batch = int(batch_match.group(1)) if batch_match else 0
        
        # Look for CSV files - they might be directly in this directory
        correct_csv = os.path.join(cp_dir, 'correct_gi_analysis.csv')
        
        # If not found, they might be in a subdirectory
        if not os.path.exists(correct_csv):
            for subdir in os.listdir(cp_dir):
                subdir_path = os.path.join(cp_dir, subdir)
                if os.path.isdir(subdir_path):
                    test_csv = os.path.join(subdir_path, 'correct_gi_analysis.csv')
                    if os.path.exists(test_csv):
                        correct_csv = test_csv
                        break
        
        has_correct = os.path.exists(correct_csv)
        
        checkpoints.append({
            'name': cp_name,
            'epoch': epoch,
            'batch': batch,
            'dir': cp_dir,
            'correct_csv': correct_csv if has_correct else None,
            'has_correct': has_correct
        })

    # Sort by epoch and then by batch
    checkpoints.sort(key=lambda x: (x['epoch'], x['batch']))

    # Load GI data from checkpoints
    gi_data = []
    for cp_info in tqdm(checkpoints):
        data = {
            'epoch': cp_info['epoch'],
            'batch': cp_info['batch'],
            'name': cp_info['name']
        }
        
        if cp_info['has_correct']:
            try:
                correct_df = pd.read_csv(cp_info['correct_csv'])
                
                # Look specifically for mean_gi which represents train-to-train GI
                if 'mean_gi' in correct_df.columns:
                    data['train_to_train_mean_gi'] = correct_df['mean_gi'].mean()
                    
                # Get percentage of GIs > 1.0 for train-to-train paths
                if 'percent_gi_above_1' in correct_df.columns:
                    data['train_to_train_percent_above_1'] = correct_df['percent_gi_above_1'].mean()
                
                # Find average of individual GIs
                gi_cols = [col for col in correct_df.columns if col.startswith('gi_')]
                if gi_cols:
                    all_gis = np.array([correct_df[col].values for col in gi_cols]).flatten()
                    all_gis = all_gis[~np.isnan(all_gis)]  # Remove NaN values
                    data['train_to_train_avg_gi'] = np.mean(all_gis)
                    data['train_to_train_percent_gi_above_1'] = 100 * np.sum(all_gis > 1.0) / len(all_gis)
            except Exception as e:
                print(f"Error loading data from {cp_info['name']}: {e}")
                data['train_to_train_mean_gi'] = None
                data['train_to_train_percent_above_1'] = None
                data['train_to_train_avg_gi'] = None
                data['train_to_train_percent_gi_above_1'] = None
        else:
            data['train_to_train_mean_gi'] = None
            data['train_to_train_percent_above_1'] = None
            data['train_to_train_avg_gi'] = None
            data['train_to_train_percent_gi_above_1'] = None
        
        gi_data.append(data)

    # Convert to DataFrame
    gi_df = pd.DataFrame(gi_data)

    # Drop rows with no GI data
    filtered_df = gi_df.dropna(subset=['train_to_train_mean_gi', 'train_to_train_avg_gi'], how='all')

    # Create sequential epoch numbering for x-axis
    filtered_df = filtered_df.sort_values(['epoch', 'batch']).reset_index(drop=True)
    filtered_df['epoch_index'] = np.arange(len(filtered_df))

    print(f"Loaded data for {len(filtered_df)} checkpoints")

    # Create the panel B plot
    fig, ax1 = plt.subplots(figsize=(10, 6), facecolor='white')

    # Main data line - train to train GI growth
    line1 = ax1.plot(filtered_df['epoch_index'], filtered_df['train_to_train_mean_gi'], 
                'o-', color='#1f77b4', linewidth=2.5, markersize=8, 
                label='Mean GI Between Training Samples')[0]
    
    # Make sure to set ylabel for the primary axis
    ax1.set_ylabel('Mean GI Between Training Samples') #, fontsize=14)

    # Create secondary y-axis for percentage above 1.0
    # Change color from orange (#ff7f0e) to a dark purple (#7030A0)
    percent_color = '#006400'  # Dark Green
    
    ax2 = ax1.twinx()
    ax2.spines['right'].set_visible(True)
    line2 = ax2.plot(filtered_df['epoch_index'], filtered_df['train_to_train_percent_above_1'], 
            'd-', color=percent_color, linewidth=2, markersize=7, label='% GI > 1.0')[0]
    ax2.set_ylabel('% of GIs > 1.0 (%)', color=percent_color) #, fontsize=14)
    ax2.tick_params(axis='y', colors=percent_color)

    # Add a reference line at GI = 1.0
    line3 = ax1.axhline(y=1.0, color='gray', linestyle='--', alpha=0.7, label='GI = 1.0 threshold')

    # Create standardized x-axis labels
    x_labels = create_epoch_labels(len(filtered_df))
    ax1.set_xticks(range(len(filtered_df)))
    ax1.set_xticklabels(x_labels)

    # Add labels and title
    ax1.set_xlabel('Epoch') #, fontsize=14)
    ax1.set_title('Growth of Generalization Intervals Between Training Samples') #, fontsize=16)

    # Create a single combined legend - avoid duplicates by listing specific lines
    ax1.legend([line1, line2, line3], 
              ['Mean GI Between Training Samples', '% GI > 1.0', 'GI = 1.0 threshold'], 
              loc='lower right')

    ax1.grid(axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    
    # Save as PDF instead of PNG
    plt.savefig(os.path.join(directory, 'panel_b_train_train_gi_growth.pdf'), 
               bbox_inches='tight', facecolor='white')
    
    # Also save as PNG as a backup
    plt.savefig(os.path.join(directory, 'panel_b_train_train_gi_growth.png'), 
               dpi=300, bbox_inches='tight', facecolor='white')
    
    plt.show()

# Usage
analysis_output_dir = 'experiments/results/results_mlp_mnist_train_gi_analysis_all_class_epoch0_10_ckpts_10'
plot_train_train_gi(analysis_output_dir)

In [ ]:
def plot_rsv_gi_evolution(results_dir):
    """Plot the evolution of GIs along RSV directions."""
    
    # Find all checkpoint directories
    checkpoint_dirs = []
    for item in os.listdir(results_dir):
        item_path = os.path.join(results_dir, item)
        if os.path.isdir(item_path) and item.startswith('model_epoch'):
            checkpoint_dirs.append(item_path)

    # Store checkpoint info
    checkpoints = []
    for cp_dir in checkpoint_dirs:
        cp_name = os.path.basename(cp_dir)
        
        # Extract epoch and batch numbers
        epoch_match = re.search(r'epoch(\d+)', cp_name)
        batch_match = re.search(r'batch(\d+)', cp_name)
        
        epoch_num = int(epoch_match.group(1)) if epoch_match else None
        batch_num = int(batch_match.group(1)) if batch_match else None
        
        # Calculate training progress
        if epoch_num is not None and batch_num is not None:
            # Assuming 400 batches per epoch (adjust if different)
            progress = epoch_num + (batch_num / 400)
        else:
            progress = None
        
        # Look for RSV GI CSV files
        correct_csv = os.path.join(cp_dir, 'correct_rsv_gi_analysis.csv')
        
        # If not found in main directory, check subdirectories
        if not os.path.exists(correct_csv):
            # Search for the file in subdirectories
            csv_files = glob.glob(os.path.join(cp_dir, "**", "correct_rsv_gi_analysis.csv"), recursive=True)
            if csv_files:
                correct_csv = csv_files[0]
        
        has_correct = os.path.exists(correct_csv)
        
        checkpoints.append({
            'name': cp_name,
            'epoch': epoch_num,
            'batch': batch_num,
            'progress': progress,
            'dir': cp_dir,
            'correct_csv': correct_csv if has_correct else None,
            'has_correct': has_correct
        })

    # Sort checkpoints by training progress
    checkpoints.sort(key=lambda x: (x['progress'] if x['progress'] is not None else float('inf')))

    # Load data from each checkpoint
    rsv_gi_data = []
    for cp_info in tqdm(checkpoints):
        data = {
            'epoch': cp_info['epoch'],
            'batch': cp_info['batch'],
            'progress': cp_info['progress'],
            'name': cp_info['name']
        }
        
        # Load correct samples GI data
        if cp_info['has_correct']:
            try:
                correct_df = pd.read_csv(cp_info['correct_csv'])
                
                # Extract RSV GI columns
                pos_gi_cols = [col for col in correct_df.columns if col.startswith('pos_gi_')]
                neg_gi_cols = [col for col in correct_df.columns if col.startswith('neg_gi_')]
                total_gi_cols = [col for col in correct_df.columns if col.startswith('total_gi_')]
                
                # Process individual GI values
                if pos_gi_cols:
                    all_pos_gis = []
                    for _, row in correct_df.iterrows():
                        for col in pos_gi_cols:
                            if not pd.isna(row[col]):
                                all_pos_gis.append(row[col])
                    data['correct_avg_pos_gi'] = np.mean(all_pos_gis) if all_pos_gis else None
                
                if neg_gi_cols:
                    all_neg_gis = []
                    for _, row in correct_df.iterrows():
                        for col in neg_gi_cols:
                            if not pd.isna(row[col]):
                                all_neg_gis.append(row[col])
                    data['correct_avg_neg_gi'] = np.mean(all_neg_gis) if all_neg_gis else None
                
                if total_gi_cols:
                    all_total_gis = []
                    for _, row in correct_df.iterrows():
                        for col in total_gi_cols:
                            if not pd.isna(row[col]):
                                all_total_gis.append(row[col])
                    data['correct_avg_total_gi'] = np.mean(all_total_gis) if all_total_gis else None
                    
                    # Also calculate percentage above 1.0
                    data['correct_pct_gi_above_1'] = 100 * np.sum(np.array(all_total_gis) > 1.0) / len(all_total_gis) if all_total_gis else None
            except Exception as e:
                print(f"Error loading data from {cp_info['name']}: {e}")
                data['correct_avg_pos_gi'] = None
                data['correct_avg_neg_gi'] = None
                data['correct_avg_total_gi'] = None
                data['correct_pct_gi_above_1'] = None
        else:
            data['correct_avg_pos_gi'] = None
            data['correct_avg_neg_gi'] = None
            data['correct_avg_total_gi'] = None
            data['correct_pct_gi_above_1'] = None
        
        rsv_gi_data.append(data)

    # Convert to DataFrame
    rsv_df = pd.DataFrame(rsv_gi_data)

    # Filter out rows with missing data and sort by progress
    filtered_rsv_df = rsv_df.dropna(subset=['correct_avg_total_gi']).sort_values('progress').reset_index(drop=True)

    # Get unique epoch values (rounding progress to nearest integer)
    epoch_values = sorted(set(np.round(filtered_rsv_df['progress']).astype(int)))
    
    # Create the plot for Panel C
    fig, ax1 = plt.subplots(figsize=(10, 6), facecolor='white')

    # Use a clear, distinguishable color palette
    pos_color = '#1f77b4'  # Blue
    neg_color = '#d62728'  # Red
    total_color = '#ff7f0e'  # Orange
    threshold_color = '#7f7f7f'  # Gray
    pct_color = '#006400'  # Dark Green

    # Apply different markers and line styles to better distinguish lines
    pos_line, = ax1.plot(filtered_rsv_df['progress'], filtered_rsv_df['correct_avg_pos_gi'], 
                        'o-', color=pos_color, linewidth=2.5, markersize=8, label='Positive GI')
    neg_line, = ax1.plot(filtered_rsv_df['progress'], filtered_rsv_df['correct_avg_neg_gi'], 
                        's--', color=neg_color, linewidth=2.5, markersize=8, label='Negative GI')
    total_line, = ax1.plot(filtered_rsv_df['progress'], filtered_rsv_df['correct_avg_total_gi'], 
                         '^-', color=total_color, linewidth=3, markersize=8, label='Total GI')
    threshold_line = ax1.axhline(y=1.0, color=threshold_color, linestyle='-.', alpha=0.7, label='GI = 1.0 threshold')

    # Set up the second y-axis
    ax2 = ax1.twinx()
    ax2.spines['right'].set_visible(True)  # Make right y-axis visible
    pct_line, = ax2.plot(filtered_rsv_df['progress'], filtered_rsv_df['correct_pct_gi_above_1'], 
                        'D:', color=pct_color, linewidth=2, markersize=7, label='% GI > 1.0')
    ax2.set_ylabel('Percentage of GIs > 1.0 (%)', color=pct_color, fontsize=14)
    ax2.tick_params(axis='y', colors=pct_color)

    # Create standardized x-axis labels
    x_labels = create_epoch_labels(len(epoch_values))
    ax1.set_xticks(epoch_values)
    ax1.set_xticklabels(x_labels)
    ax1.set_xlim(-0.2, max(epoch_values) + 0.2)

    # Add grid lines
    ax1.grid(axis='y', linestyle='--', alpha=0.3, color='#cccccc')

    # Add labels and title
    ax1.set_xlabel('Training Epoch') #, fontsize=14)
    ax1.set_ylabel('Average RSV GI') #, fontsize=14)
    ax1.set_title('Evolution of Generalization Intervals (GIs) Along RSV Directions') #, fontsize=16)

    # Create legend with explicit handles and labels
    ax1.legend([pos_line, neg_line, total_line, threshold_line, pct_line], 
              ['Positive GI', 'Negative GI', 'Total GI', 'GI = 1.0 threshold', '% GI > 1.0'], 
              loc='upper left', frameon=True, facecolor='white', edgecolor='lightgray')

    plt.tight_layout()
        # Save as PDF instead of PNG
    plt.savefig(os.path.join(directory, 'panel_c_rsv_gi_evolution.pdf'), 
               bbox_inches='tight', facecolor='white')
    plt.savefig(os.path.join(directory, 'panel_c_rsv_gi_evolution.png'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

# Usage
results_dir = "experiments/results/results_mlp_mnist_gi_evolution_epoch0_batch0_to_epoch9_batch399"
plot_rsv_gi_evolution(results_dir)

In [ ]:
def plot_random_gi_evolution(results_dir):
    """Plot the evolution of GIs along random directions."""
    
    # Find all checkpoint directories
    checkpoint_dirs = []
    for item in os.listdir(results_dir):
        item_path = os.path.join(results_dir, item)
        if os.path.isdir(item_path) and item.startswith('model_epoch'):
            checkpoint_dirs.append(item_path)

    # Store checkpoint info
    checkpoints = []
    for cp_dir in checkpoint_dirs:
        cp_name = os.path.basename(cp_dir)
        
        # Extract epoch and batch numbers
        epoch_match = re.search(r'epoch(\d+)', cp_name)
        batch_match = re.search(r'batch(\d+)', cp_name)
        
        epoch_num = int(epoch_match.group(1)) if epoch_match else None
        batch_num = int(batch_match.group(1)) if batch_match else None
        
        # Look for Random Direction GI CSV files
        correct_csv = os.path.join(cp_dir, 'correct_random_gi_analysis.csv')
        
        # If not found in main directory, check subdirectories
        if not os.path.exists(correct_csv):
            csv_files = glob.glob(os.path.join(cp_dir, "**", "correct_random_gi_analysis.csv"), recursive=True)
            if csv_files:
                correct_csv = csv_files[0]
        
        has_correct = os.path.exists(correct_csv)
        
        checkpoints.append({
            'name': cp_name,
            'epoch': epoch_num,
            'batch': batch_num,
            'dir': cp_dir,
            'correct_csv': correct_csv if has_correct else None,
            'has_correct': has_correct
        })

    # Sort checkpoints by epoch and then by batch
    checkpoints.sort(key=lambda x: (x['epoch'], x['batch']))

    # Load random direction GI data
    random_gi_data = []
    for cp_info in tqdm(checkpoints):
        data = {
            'epoch': cp_info['epoch'],
            'batch': cp_info['batch'],
            'name': cp_info['name']
        }
        
        if cp_info['has_correct']:
            try:
                correct_df = pd.read_csv(cp_info['correct_csv'])
                
                # Individual GI values by type
                pos_gi_cols = [col for col in correct_df.columns if col.startswith('pos_gi_')]
                neg_gi_cols = [col for col in correct_df.columns if col.startswith('neg_gi_')]
                total_gi_cols = [col for col in correct_df.columns if col.startswith('total_gi_') or col.startswith('gi_')]
                
                # Process positive GI values
                if pos_gi_cols:
                    all_pos_gis = []
                    for _, row in correct_df.iterrows():
                        for col in pos_gi_cols:
                            if not pd.isna(row[col]):
                                all_pos_gis.append(row[col])
                    
                    data['random_avg_indiv_pos_gi'] = np.mean(all_pos_gis) if all_pos_gis else None
                    data['random_percent_pos_gi_above_1'] = 100 * np.sum(np.array(all_pos_gis) > 1.0) / len(all_pos_gis) if all_pos_gis else None
                
                # Process negative GI values
                if neg_gi_cols:
                    all_neg_gis = []
                    for _, row in correct_df.iterrows():
                        for col in neg_gi_cols:
                            if not pd.isna(row[col]):
                                all_neg_gis.append(row[col])
                    
                    data['random_avg_indiv_neg_gi'] = np.mean(all_neg_gis) if all_neg_gis else None
                    data['random_percent_neg_gi_above_1'] = 100 * np.sum(np.array(all_neg_gis) > 1.0) / len(all_neg_gis) if all_neg_gis else None
                
                # Process total GI values
                if total_gi_cols:
                    all_total_gis = []
                    for _, row in correct_df.iterrows():
                        for col in total_gi_cols:
                            if not pd.isna(row[col]):
                                all_total_gis.append(row[col])
                    
                    data['random_avg_indiv_total_gi'] = np.mean(all_total_gis) if all_total_gis else None
                    data['random_percent_total_gi_above_1'] = 100 * np.sum(np.array(all_total_gis) > 1.0) / len(all_total_gis) if all_total_gis else None
            except Exception as e:
                print(f"Error loading data from {cp_info['name']}: {e}")
                for key in ['random_avg_indiv_pos_gi', 'random_avg_indiv_neg_gi', 'random_avg_indiv_total_gi',
                          'random_percent_pos_gi_above_1', 'random_percent_neg_gi_above_1', 'random_percent_total_gi_above_1']:
                    data[key] = None
        else:
            for key in ['random_avg_indiv_pos_gi', 'random_avg_indiv_neg_gi', 'random_avg_indiv_total_gi',
                      'random_percent_pos_gi_above_1', 'random_percent_neg_gi_above_1', 'random_percent_total_gi_above_1']:
                data[key] = None
        
        random_gi_data.append(data)

    # Convert to DataFrame
    random_df = pd.DataFrame(random_gi_data)

    # Filter and sort
    filtered_df = random_df.dropna(subset=['random_avg_indiv_total_gi'], how='all')
    filtered_df = filtered_df.sort_values(['epoch', 'batch']).reset_index(drop=True)
    filtered_df['epoch_index'] = np.arange(len(filtered_df))

    print(f"Loaded data for {len(filtered_df)} checkpoints")

    # Create the plot for Panel D
    fig, ax1 = plt.subplots(figsize=(10, 6), facecolor='white')

    # Use different colors and line styles for clear distinction
    pos_color = '#1f77b4'  # Blue
    neg_color = '#d62728'  # Red
    total_color = '#ff7f0e'  # Orange
    pct_color = '#006400'  # Dark Green

    # Plot all three GI types with different markers and styles
    pos_line, = ax1.plot(filtered_df['epoch_index'], filtered_df['random_avg_indiv_pos_gi'], 
                        'o-', color=pos_color, linewidth=2.5, markersize=8, label='Positive GI')
    neg_line, = ax1.plot(filtered_df['epoch_index'], filtered_df['random_avg_indiv_neg_gi'], 
                        's--', color=neg_color, linewidth=2.5, markersize=8, label='Negative GI')
    total_line, = ax1.plot(filtered_df['epoch_index'], filtered_df['random_avg_indiv_total_gi'], 
                         '^-', color=total_color, linewidth=2.5, markersize=8, label='Total GI')

    # Add reference line at GI = 1.0
    threshold_line = ax1.axhline(y=1.0, color='gray', linestyle='-.', alpha=0.7, label='GI = 1.0 threshold')

    # Create secondary y-axis for percentage above 1.0
    ax2 = ax1.twinx()
    ax2.spines['right'].set_visible(True)
    pct_line, = ax2.plot(filtered_df['epoch_index'], filtered_df['random_percent_total_gi_above_1'], 
                        'D:', color=pct_color, linewidth=2, markersize=7, label='% GI > 1.0')
    ax2.set_ylabel('Percentage of GIs > 1.0 (%)', color=pct_color) #, fontsize=14)
    ax2.tick_params(axis='y', colors=pct_color)

    # Create standardized x-axis labels
    x_labels = create_epoch_labels(len(filtered_df))
    ax1.set_xticks(range(len(filtered_df)))
    ax1.set_xticklabels(x_labels)
    ax1.set_xlim(-0.2, len(filtered_df) - 0.8)

    # Add grid lines
    ax1.grid(axis='y', linestyle='--', alpha=0.3, color='#cccccc')

    # Add labels and title
    ax1.set_xlabel('Epoch') # fontsize=14)
    ax1.set_ylabel('Average GI Along Random Directions') #, fontsize=14)
    ax1.set_title('Evolution of Generalization Intervals Along Random Directions') #, fontsize=16)

    # Create a single combined legend with all lines from both axes
    all_lines = [pos_line, neg_line, total_line, threshold_line, pct_line]
    all_labels = ['Positive GI', 'Negative GI', 'Total GI', 'GI = 1.0 threshold', '% GI > 1.0']
    ax1.legend(all_lines, all_labels, loc='upper left',framealpha=0.9) # fontsize=12, )

    plt.tight_layout()
    plt.savefig(os.path.join(directory, 'panel_d_random_gi_evolution.pdf'), 
               bbox_inches='tight', facecolor='white')
    plt.savefig(os.path.join(directory, 'panel_d_random_gi_evolution.png'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

# Usage
results_dir = "experiments/results/results_mlp_mnist_gi_evolution_epoch0_batch0_to_epoch9_batch399"
plot_random_gi_evolution(results_dir)

In [ ]:
def create_combined_plots():
    """Create a combined figure with three panels in a row."""
    
    # Define a horizontal rectangle figure for 1x3 layout
    plt.figure(figsize=(18, 6), facecolor='white')
    
    # Load saved images
    panel_b_img = mpimg.imread(os.path.join(directory, 'panel_b_train_train_gi_growth.png'))
    panel_c_img = mpimg.imread(os.path.join(directory, 'panel_c_rsv_gi_evolution.png'))
    panel_d_img = mpimg.imread(os.path.join(directory, 'panel_d_random_gi_evolution.png'))
    
    # Create 1x3 subplots with titles under the plots
    plt.subplot(131)
    plt.imshow(panel_b_img)
    plt.axis('off')
    plt.title('') #, fontsize=18)  # Empty title at top
    plt.xlabel('A: Train-Train GI',labelpad=20) # fontsize=18, )  # Title below plot
    
    plt.subplot(132)
    plt.imshow(panel_c_img)
    plt.axis('off')
    plt.title('') #, fontsize=18)  # Empty title at top
    plt.xlabel('B: RSV Direction GI', labelpad=20) #fontsize=18, )  # Title below plot
    
    plt.subplot(133)
    plt.imshow(panel_d_img)
    plt.axis('off')
    plt.title('') #, fontsize=18)  # Empty title at top
    plt.xlabel('C: Random Direction GI',  labelpad=20) # fontsize=18,)  # Title below plot
    
    # Adjust layout
    plt.tight_layout()
    
    # Save the combined figure as both PDF and PNG
    plt.savefig(os.path.join(directory, 'combined_evolution_plots_1x3.pdf'), 
               bbox_inches='tight', facecolor='white')
    plt.savefig(os.path.join(directory, 'combined_evolution_plots_1x3.png'), 
               dpi=300, bbox_inches='tight', facecolor='white')
    
    plt.show()
    
    print("Combined figure saved as 'combined_evolution_plots_1x3.pdf' and 'combined_evolution_plots_1x3.png'")

# Create the combined plot after generating all individual panels
create_combined_plots()

## RSV vs Random Direction Growth

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import re
from tqdm import tqdm

# Set clean, modern plotting style
plt.style.use('default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Liberation Sans']
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Directories containing the checkpoint data
results_dir = "experiments/results/results_mlp_mnist_gi_evolution_epoch0_batch0_to_epoch9_batch399"

def extract_checkpoint_info(checkpoint_name):
    """Extract epoch and batch numbers from checkpoint name."""
    epoch_match = re.search(r'epoch(\d+)', checkpoint_name)
    batch_match = re.search(r'batch(\d+)', checkpoint_name)
    
    epoch_num = int(epoch_match.group(1)) if epoch_match else None
    batch_num = int(batch_match.group(1)) if batch_match else None
    
    return epoch_num, batch_num

# Function to load checkpoint data for both RSV and random directions
def load_checkpoint_data(results_dir):
    # Find all checkpoint directories
    checkpoint_dirs = []
    for item in os.listdir(results_dir):
        item_path = os.path.join(results_dir, item)
        if os.path.isdir(item_path) and item.startswith('model_epoch'):
            checkpoint_dirs.append(item_path)
    
    # Store checkpoint info
    checkpoints = []
    for cp_dir in checkpoint_dirs:
        cp_name = os.path.basename(cp_dir)
        epoch_num, batch_num = extract_checkpoint_info(cp_name)
        
        # Look for RSV GI CSV files
        rsv_csv = os.path.join(cp_dir, 'correct_rsv_gi_analysis.csv')
        if not os.path.exists(rsv_csv):
            rsv_files = glob.glob(os.path.join(cp_dir, "**", "correct_rsv_gi_analysis.csv"), recursive=True)
            if rsv_files:
                rsv_csv = rsv_files[0]
        
        # Look for Random Direction GI CSV files
        random_csv = os.path.join(cp_dir, 'correct_random_gi_analysis.csv')
        if not os.path.exists(random_csv):
            random_files = glob.glob(os.path.join(cp_dir, "**", "correct_random_gi_analysis.csv"), recursive=True)
            if random_files:
                random_csv = random_files[0]
        
        has_rsv = os.path.exists(rsv_csv)
        has_random = os.path.exists(random_csv)
        
        checkpoints.append({
            'name': cp_name,
            'epoch': epoch_num,
            'batch': batch_num,
            'dir': cp_dir,
            'rsv_csv': rsv_csv if has_rsv else None,
            'random_csv': random_csv if has_random else None,
            'has_rsv': has_rsv,
            'has_random': has_random
        })
    
    # Sort by epoch and batch
    checkpoints.sort(key=lambda x: (x['epoch'], x['batch']))
    
    return checkpoints

# Function to load GI data from both RSV and random direction CSVs
def load_gi_data(checkpoint_info):
    data = {
        'epoch': checkpoint_info['epoch'],
        'batch': checkpoint_info['batch'],
        'name': checkpoint_info['name']
    }
    
    # Load RSV data
    if checkpoint_info['has_rsv']:
        try:
            rsv_df = pd.read_csv(checkpoint_info['rsv_csv'])
            
            # Extract the total GI data from RSV
            total_gi_cols = [col for col in rsv_df.columns if col.startswith('total_gi_')]
            if total_gi_cols:
                all_gis = []
                for _, row in rsv_df.iterrows():
                    for col in total_gi_cols:
                        if not pd.isna(row[col]):
                            all_gis.append(row[col])
                
                data['rsv_avg_gi'] = np.mean(all_gis) if all_gis else None
            else:
                # Alternative column names
                if 'mean_total_gi' in rsv_df.columns:
                    data['rsv_avg_gi'] = rsv_df['mean_total_gi'].mean()
                elif 'mean_gi' in rsv_df.columns:
                    data['rsv_avg_gi'] = rsv_df['mean_gi'].mean()
                else:
                    data['rsv_avg_gi'] = None
        except Exception as e:
            print(f"Error loading RSV data from {checkpoint_info['name']}: {e}")
            data['rsv_avg_gi'] = None
    else:
        data['rsv_avg_gi'] = None
    
    # Load Random direction data
    if checkpoint_info['has_random']:
        try:
            random_df = pd.read_csv(checkpoint_info['random_csv'])
            
            # Extract the total GI data from Random directions
            total_gi_cols = [col for col in random_df.columns if col.startswith('total_gi_')]
            if total_gi_cols:
                all_gis = []
                for _, row in random_df.iterrows():
                    for col in total_gi_cols:
                        if not pd.isna(row[col]):
                            all_gis.append(row[col])
                
                data['random_avg_gi'] = np.mean(all_gis) if all_gis else None
            else:
                # Alternative column names
                if 'mean_total_gi' in random_df.columns:
                    data['random_avg_gi'] = random_df['mean_total_gi'].mean()
                elif 'mean_gi' in random_df.columns:
                    data['random_avg_gi'] = random_df['mean_gi'].mean()
                else:
                    data['random_avg_gi'] = None
        except Exception as e:
            print(f"Error loading Random data from {checkpoint_info['name']}: {e}")
            data['random_avg_gi'] = None
    else:
        data['random_avg_gi'] = None
    
    return data

# Load all checkpoint data
print("Loading GI data for RSV and Random directions...")
checkpoints = load_checkpoint_data(results_dir)
gi_data = []
for cp_info in tqdm(checkpoints):
    gi_data.append(load_gi_data(cp_info))

# Convert to DataFrame
gi_df = pd.DataFrame(gi_data)

# Filter out rows with missing data
filtered_df = gi_df.dropna(subset=['rsv_avg_gi', 'random_avg_gi'], how='all')
filtered_df = filtered_df.sort_values(['epoch', 'batch']).reset_index(drop=True)

# Create epoch index for x-axis
filtered_df['epoch_index'] = np.arange(len(filtered_df))

# Calculate relative growth from first checkpoint
first_checkpoint = filtered_df.iloc[0]
if 'rsv_avg_gi' in first_checkpoint and first_checkpoint['rsv_avg_gi'] > 0:
    filtered_df['rsv_relative_growth'] = 100 * (filtered_df['rsv_avg_gi'] / first_checkpoint['rsv_avg_gi'] - 1)
else:
    filtered_df['rsv_relative_growth'] = None

if 'random_avg_gi' in first_checkpoint and first_checkpoint['random_avg_gi'] > 0:
    filtered_df['random_relative_growth'] = 100 * (filtered_df['random_avg_gi'] / first_checkpoint['random_avg_gi'] - 1)
else:
    filtered_df['random_relative_growth'] = None

# Create the comparison plot
plt.figure(figsize=(10, 6))

# Plot relative growth for RSV and Random directions
plt.plot(filtered_df['epoch_index'], filtered_df['rsv_relative_growth'], 'o-', 
         color='#1f77b4', linewidth=2.5, markersize=8, label='RSV Directions')
plt.plot(filtered_df['epoch_index'], filtered_df['random_relative_growth'], 's-', 
         color='#006400', linewidth=2.5, markersize=8, label='Random Directions')

# Add a horizontal line at 0% growth
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.7)

# Create x-tick labels
x_tick_labels = ['Initial'] + [f'Epoch {i}' for i in range(1, len(filtered_df))]
if len(x_tick_labels) > len(filtered_df):
    x_tick_labels = x_tick_labels[:len(filtered_df)]
plt.xticks(range(len(filtered_df)), x_tick_labels, rotation=45)

# Add labels and title
plt.xlabel('Epoch') #, fontsize=14)
plt.ylabel('Relative Growth from Initial Checkpoint (%)') #, fontsize=14)
plt.title('Comparison of GI Growth: RSV vs. Random Directions') #, fontsize=16)

# Add grid lines
plt.grid(axis='y', linestyle='--', alpha=0.3, color='#cccccc')

# Add legend
plt.legend(loc='upper left')

plt.tight_layout()
plt.savefig(os.path.join(directory, 'rsv_vs_random_growth_comparison.pdf'), 
               bbox_inches='tight', facecolor='white')
plt.savefig(os.path.join(directory, 'rsv_vs_random_growth_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

print("Comparison plot has been saved as 'rsv_vs_random_growth_comparison.png'")